# 02 - Feature Engineering

This notebook converts the warehouse extract into a model-ready dataset with leakage controls, derived banking ratios, and a temporal train-test split.

In [1]:
from pathlib import Path
import sys

def find_ml_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / '4_ml' / 'src').exists():
            return candidate / '4_ml'
        if candidate.name == '4_ml' and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the 4_ml workspace.')

ML_ROOT = find_ml_root()
if str(ML_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ML_ROOT / 'src'))

import pandas as pd

from config import OUTPUT_DIR, RISK_TARGET_THRESHOLD
from data_access import load_feature_frame
from features import build_model_frame, build_targets, engineer_features, select_feature_columns, split_temporal

raw_frame = load_feature_frame()
model_frame = build_model_frame(raw_frame)
model_frame.head()

,customer_id,scoring_date,load_date,customer_tenure_days,account_count,total_working_balance,avg_working_balance,negative_balance_account_count,oldest_account_opening_date,newest_account_opening_date,...,days_since_last_kyc_review,days_until_next_kyc_review,balance_per_account,negative_balance_rate,salary_to_balance_ratio,has_overdraft,is_new_customer,is_kyc_stale,is_kyc_very_stale,client_nature_flag
0,113336184,2026-05-18,2026-05-18,3058,1.0,2.088,2.088,0.0,2018-01-02,2018-01-02,...,0.0,0.0,2.088,0.0,0.000000,0,0,0,0,PPH
1,113199291,2026-05-18,2026-05-18,3058,1.0,-50.808,-50.808,1.0,2018-01-02,2018-01-02,...,0.0,0.0,-50.808,1.0,-22.142182,1,0,0,0,PPH
2,113318555,2026-05-18,2026-05-18,3058,1.0,22900.274,22900.274,0.0,2018-01-02,2018-01-02,...,0.0,0.0,22900.274,0.0,0.000000,0,0,0,0,PPH
3,113160985,2026-05-18,2026-05-18,3058,NaN,NaN,NaN,NaN,NaT,NaT,...,0.0,0.0,0.000,0.0,0.000000,0,0,0,0,PPH
4,113182752,2026-05-18,2026-05-18,3058,NaN,NaN,NaN,NaN,NaT,NaT,...,0.0,0.0,0.000,0.0,0.000000,0,0,0,0,PPH


In [2]:
engineered = build_targets(engineer_features(raw_frame))
feature_columns, numeric_columns, categorical_columns = select_feature_columns(engineered)
target_columns = ['risk_target', 'risk_probability_target']
if 'churn_target' in engineered.columns and engineered['churn_target'].notna().any():
    target_columns.append('churn_target')

feature_columns[:20], numeric_columns[:20], categorical_columns[:20]

(['customer_tenure_days',
  'account_count',
  'total_working_balance',
  'avg_working_balance',
  'negative_balance_account_count',
  'monthly_salary',
  'number_of_dependents',
  'days_since_last_kyc_review',
  'days_until_next_kyc_review',
  'balance_per_account',
  'negative_balance_rate',
  'salary_to_balance_ratio',
  'has_overdraft',
  'is_new_customer',
  'is_kyc_stale',
  'is_kyc_very_stale',
  'is_kyc_complete',
  'is_pep',
  'is_compliance_flagged',
  'has_compliance_decision'],
 ['customer_tenure_days',
  'account_count',
  'total_working_balance',
  'avg_working_balance',
  'negative_balance_account_count',
  'monthly_salary',
  'number_of_dependents',
  'days_since_last_kyc_review',
  'days_until_next_kyc_review',
  'balance_per_account',
  'negative_balance_rate',
  'salary_to_balance_ratio',
  'has_overdraft',
  'is_new_customer',
  'is_kyc_stale',
  'is_kyc_very_stale',
  'is_kyc_complete',
  'is_pep',
  'is_compliance_flagged',
  'has_compliance_decision'],
 ['employm

In [3]:
train_frame, test_frame = split_temporal(engineered, date_column='scoring_date', test_fraction=0.2)
train_frame['risk_target'].value_counts(normalize=True).rename('train_share'), test_frame['risk_target'].value_counts(normalize=True).rename('test_share')

(risk_target
 0    0.99026
 1    0.00974
 Name: train_share, dtype: float64,
 risk_target
 0    0.972673
 1    0.027327
 Name: test_share, dtype: float64)

In [4]:
export_dir = OUTPUT_DIR / 'feature_engineering'
export_dir.mkdir(parents=True, exist_ok=True)

train_path = export_dir / 'train_frame.csv'
test_path = export_dir / 'test_frame.csv'
feature_list_path = export_dir / 'feature_columns.txt'

train_frame.to_csv(train_path, index=False)
test_frame.to_csv(test_path, index=False)
feature_list_path.write_text('\n'.join(feature_columns), encoding='utf-8')

{
    'train_rows': len(train_frame),
    'test_rows': len(test_frame),
    'feature_count': len(feature_columns),
    'target_threshold': RISK_TARGET_THRESHOLD,
    'train_path': str(train_path),
    'test_path': str(test_path)
}

{'train_rows': 109340,
 'test_rows': 27336,
 'feature_count': 37,
 'target_threshold': 50.0,
 'train_path': 'C:\\Users\\Ahmed\\Desktop\\atb_bi_project\\4_ml\\outputs\\feature_engineering\\train_frame.csv',
 'test_path': 'C:\\Users\\Ahmed\\Desktop\\atb_bi_project\\4_ml\\outputs\\feature_engineering\\test_frame.csv'}

## Feature set summary

The engineered set keeps base banking drivers and time-derived features, while excluding direct leakage columns such as the rule-based risk scores and tier labels from the model feature list.

For churn, the notebook keeps a placeholder column so a future label can be attached without changing the downstream pipeline contract.